In [1]:
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.9/395.9 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 242.7/242.7 kB 18.2 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import joblib
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
import optuna
from tqdm.auto import tqdm
import warnings
import holidays
import os

warnings.filterwarnings('ignore')


try:
    from google.colab import drive
    drive.mount('/content/drive')
    PATH = "/content/drive/MyDrive/열수요예측/"
    if not os.path.exists(PATH):
        os.makedirs(PATH)
except ImportError:
    PATH = "./"


SEED = 42
np.random.seed(SEED)


def load_and_preprocess(df):
    """기본 컬럼명 정리 및 타입 변환 (tm 컬럼 자동 탐지 기능 추가)"""
    df = df.copy()
    if 'Unnamed: 0' in df.columns:
        df = df.drop(columns=['Unnamed: 0'])

    df.columns = [col.lower() for col in df.columns]


    df.columns = [col.replace('train_heat.', '').replace('test_heat.', '') for col in df.columns]

    if 'tm' not in df.columns:
        time_col = [col for col in df.columns if 'tm' in col]
        if time_col:
            print(f"'{time_col[0]}' 컬럼을 'tm' 컬럼으로 변경합니다.")
            df.rename(columns={time_col[0]: 'tm'}, inplace=True)
        else:
            print(f"현재 데이터프레임의 컬럼 목록: {df.columns.tolist()}")
            raise KeyError("데이터에서 'tm' 또는 'tm'을 포함하는 시간 관련 컬럼을 찾을 수 없습니다.")


    if df['tm'].dtype == 'object':
        df['tm'] = pd.to_datetime(df['tm'])
    else:
        df['tm'] = pd.to_datetime(df['tm'], format='%Y%m%d%H')


    if 'branch_id' in df.columns:
        df['branch_id'] = df['branch_id'].astype('category')


    return df

def create_missing_flags_and_replace(df):
    """결측치 플래그 생성 및 NaN으로 변환"""
    df = df.copy()
    missing_cols = ['ta', 'ws', 'rn_day', 'rn_hr1', 'hm', 'si', 'ta_chi', 'heat_demand']

    for col in missing_cols:
        if col in df.columns:
            missing_mask = (df[col] == -99)
            df[f'{col}_missing'] = missing_mask.astype(int)
            df[col] = df[col].replace(-99, np.nan)

    if 'wd' in df.columns:
        df['wd'] = df['wd'].replace(-9.9, np.nan)

    return df

def apply_interpolation(df):
    """지사별 시계열 보간"""
    df = df.copy()
    print("지사별 시계열 보간 적용 중...")


    interp_cols = ['ta', 'ws', 'rn_day', 'rn_hr1', 'hm', 'si', 'ta_chi', 'wd']
    if 'heat_demand' in df.columns:
        interp_cols.append('heat_demand')


    current_interp_cols = [col for col in interp_cols if col in df.columns]


    df = df.sort_values(by=['branch_id', 'tm']).reset_index(drop=True)


    for col in current_interp_cols:
        df[col] = df.groupby('branch_id')[col].transform(lambda group: group.interpolate(method='linear', limit_direction='both'))


    remaining_nan_before_fill = df[current_interp_cols].isnull().sum().sum()
    if remaining_nan_before_fill > 0:
        print(f"경고: 보간 후에도 {remaining_nan_before_fill}개의 결측치가 남아있습니다. 0으로 채웁니다.")


        cols_to_fill_zero = [col for col in current_interp_cols if col != 'branch_id']
        df[cols_to_fill_zero] = df[cols_to_fill_zero].fillna(0)


    return df
def add_heating_season(df):
    """난방 시즌 컬럼 추가"""
    df = df.copy()
    df['month'] = df['tm'].dt.month
    df['heating_season'] = 0
    heating_months = [10, 11, 12, 1, 2, 3, 4]
    df.loc[df['month'].isin(heating_months), 'heating_season'] = 1
    return df

def create_advanced_features(df):
    """고도화된 파생변수 생성"""
    df = df.copy()


    df['year'] = df['tm'].dt.year
    df['day'] = df['tm'].dt.day
    df['hour'] = df['tm'].dt.hour
    df['dayofweek'] = df['tm'].dt.dayofweek
    df['dayofyear'] = df['tm'].dt.dayofyear

    # 순환 인코딩
    df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
    df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)
    df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
    df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)
    df['dayofweek_sin'] = np.sin(2 * np.pi * df['dayofweek'] / 7)
    df['dayofweek_cos'] = np.cos(2 * np.pi * df['dayofweek'] / 7)

    # 고급 기상 변수
    df['HDD18'] = np.maximum(0, 18 - df['ta']) # 난방도일

    # 체감온도 계산 (겨울용)
    def calculate_apparent_temp(ta, ws):
        return 13.12 + 0.6215 * ta - 11.37 * (ws * 3.6)**0.16 + 0.3965 * ta * (ws * 3.6)**0.16
    df['apparent_temp'] = calculate_apparent_temp(df['ta'], df['ws'])

    # 시차 및 이동평균 (지사별)
    for lag in [3, 6, 24]:
        if 'ta' in df.columns:
            df[f'ta_lag_{lag}h'] = df.groupby('branch_id')['ta'].shift(lag)
    for window in [6, 12, 24]:
        if 'ta' in df.columns:
            df[f'ta_ma_{window}h'] = df.groupby('branch_id')['ta'].transform(lambda x: x.rolling(window, min_periods=1).mean())
    if 'ta' in df.columns:
        df['ta_diff_3h'] = df.groupby('branch_id')['ta'].diff(3)
        df['ta_diff_6h'] = df.groupby('branch_id')['ta'].diff(6)

    # 공휴일
    kr_holidays = holidays.KR()
    df['is_holiday'] = df['tm'].dt.date.apply(lambda x: x in kr_holidays).astype(int)

    # 다시 보간 (시차 변수 생성 후 생긴 NaN 처리)
    # heat_demand가 없는 경우를 고려하여 drop할 컬럼 목록 동적 생성
    cols_to_drop = ['tm']
    if 'heat_demand' in df.columns:
        cols_to_drop.append('heat_demand')

    # 모든 컬럼을 대상으로 NaN을 처리하기 전에, drop할 컬럼만 제외
    feature_cols_for_fillna = [col for col in df.columns if col not in cols_to_drop]

    df[feature_cols_for_fillna] = df.groupby('branch_id')[feature_cols_for_fillna].transform(lambda x: x.ffill().bfill())

    # --- 수정된 코드 시작 ---
    # branch_id 컬럼을 제외하고 fillna(0) 적용
    cols_for_global_fillna = [col for col in df.columns if col not in cols_to_drop and df[col].dtype != 'category']
    df[cols_for_global_fillna] = df[cols_for_global_fillna].fillna(0)
    # --- 수정된 코드 끝 ---

    return df

def create_weather_outlier_flags(train_df, test_df):
    """시즌별 기상 이상치 플래그 생성 (훈련 세트 기준)"""
    print("시즌별 기상 이상치 플래그 생성 중...")
    outlier_thresholds = {}

    # train_df와 test_df 모두 'heating_season'과 'branch_id'를 가지고 있다고 가정
    # 또한 'ta', 'ws', 'rn_day' 컬럼이 존재한다고 가정합니다.

    for branch in train_df['branch_id'].unique():
        branch_data = train_df[train_df['branch_id'] == branch]
        outlier_thresholds[branch] = {}
        for season in [0, 1]:
            season_data = branch_data[branch_data['heating_season'] == season]
            if len(season_data) > 10:
                outlier_thresholds[branch][season] = {
                    'ta_q10': season_data['ta'].quantile(0.10), # 극한 추위
                    'ws_q90': season_data['ws'].quantile(0.90), # 강풍
                    'rn_day_q90': season_data['rn_day'].quantile(0.90) # 폭우
                }

    def apply_thresholds(df, thresholds):
        df = df.copy()
        # 이상치 플래그 컬럼 초기화 (초기화 전에 이미 존재한다면 덮어씁니다)
        df['cold_extreme'] = 0
        df['strong_wind'] = 0
        df['heavy_rain'] = 0

        for branch in df['branch_id'].unique():
            if branch in thresholds:
                branch_mask = df['branch_id'] == branch
                for season in [0, 1]:
                    if season in thresholds[branch]:
                        season_mask = branch_mask & (df['heating_season'] == season)
                        season_thresholds = thresholds[branch][season]

                        # 해당 컬럼이 데이터프레임에 있는지 확인 후 적용
                        if 'ta' in df.columns:
                            df.loc[season_mask, 'cold_extreme'] = (df.loc[season_mask, 'ta'] < season_thresholds['ta_q10']).astype(int)
                        if 'ws' in df.columns:
                            df.loc[season_mask, 'strong_wind'] = (df.loc[season_mask, 'ws'] > season_thresholds['ws_q90']).astype(int)
                        if 'rn_day' in df.columns:
                            df.loc[season_mask, 'heavy_rain'] = (df.loc[season_mask, 'rn_day'] > season_thresholds['rn_day_q90']).astype(int)
        return df

    train_result = apply_thresholds(train_df, outlier_thresholds)
    test_result = apply_thresholds(test_df, outlier_thresholds)

    return train_result, test_result

def get_feature_names(df):
    """사용할 피처 이름 정의"""
    # 원본 숫자/범주 + 결측치 플래그 + 이상치 플래그 + 파생변수
    # df.columns는 이미 소문자로 되어있을 것으로 가정
    features = [col for col in df.columns if col not in ['tm', 'heat_demand', 'year']]

    # 범주형 피처 명시
    categorical_features = [
        'branch_id', 'month', 'hour', 'dayofweek', 'heating_season', 'is_holiday',
        'cold_extreme', 'strong_wind', 'heavy_rain'
    ]
    # 실제 존재하는 범주형 피처만 필터링
    categorical_features = [f for f in categorical_features if f in df.columns]

    return features, categorical_features

# ---------------------------------------------------
# 2. 모델 훈련 및 최적화
# ---------------------------------------------------

def create_year_based_cv_splits(df):
    """연도 기반 3-Fold CV 분할 생성"""
    cv_splits = []
    years = sorted(df['year'].unique())

    # 마지막 3개년을 검증 세트로 사용
    val_years_to_use = years[-3:] if len(years) >= 3 else years # 오타 수정 반영

    if not val_years_to_use: # val_years_to_use가 비어있는 경우
        print("경고: 연도 데이터가 충분하지 않아 연도 기반 CV를 생성할 수 없습니다. 일반 KFold(3-Fold)를 사용합니다.")
        kf = KFold(n_splits=3, shuffle=True, random_state=SEED)
        cv_splits = list(kf.split(df))
        return cv_splits

    for val_year in val_years_to_use:
        train_mask = (df['year'] != val_year)
        val_mask = (df['year'] == val_year)

        train_indices = df[train_mask].index.tolist()
        val_indices = df[val_mask].index.tolist()

        # 데이터가 있는 경우에만 추가
        if len(train_indices) > 0 and len(val_indices) > 0:
            cv_splits.append((train_indices, val_indices))
            print(f"   Fold: Val Year={val_year}, Train size={len(train_indices)}, Val size={len(val_indices)}")

    if not cv_splits:
        # 데이터가 너무 적을 경우 일반 KFold로 대체 (이미 위에서 처리되었지만, 안전을 위해 남겨둠)
        print("연도 기반 CV를 생성할 데이터가 부족하여 일반 KFold(3-Fold)를 사용합니다.")
        kf = KFold(n_splits=3, shuffle=True, random_state=SEED)
        cv_splits = list(kf.split(df))

    return cv_splits

def get_monotone_constraints(feature_names):
    """LightGBM 단조성 제약 설정"""
    constraints = [0] * len(feature_names)
    for i, feature in enumerate(feature_names):
        if 'ta' in feature or 'apparent_temp' in feature:
            constraints[i] = -1 # 온도/체감온도 상승 시 열수요 감소
        elif feature == 'HDD18':
            constraints[i] = 1 # 난방도일 상승 시 열수요 증가
        elif 'ws' in feature:
            constraints[i] = 1 # 풍속 상승 시 열수요 증가
    return constraints

def optimize_and_train(df_group, group_name):
    """Optuna로 하이퍼파라미터 최적화 및 모델 훈련"""
    print(f"\n===== {group_name.upper()} 그룹 모델 훈련 시작 =====")

    # heat_demand 컬럼이 없는 경우 (테스트 데이터에서 호출 시) 예외 처리
    if 'heat_demand' not in df_group.columns:
        print(f"경고: {group_name} 그룹 데이터에 'heat_demand' 컬럼이 없습니다. 모델 훈련을 건너뜝니다.")
        return None

    X = df_group.drop(columns=['tm', 'heat_demand'])
    y = df_group['heat_demand']
    features, categorical_features = get_feature_names(X)
    X = X[features] # 정의된 features만 사용

    # X에 branch_id가 없으면 문제가 되므로 확인
    if 'branch_id' not in X.columns:
        print(f"오류: {group_name} 그룹 데이터에 'branch_id' 컬럼이 없습니다. 모델 훈련 중단.")
        return None

    # 연도 기반 CV 분할
    cv_splits = create_year_based_cv_splits(df_group)

    # CV 분할이 유효한지 확인
    if not cv_splits:
        print(f"경고: {group_name} 그룹에 유효한 CV 분할이 없습니다. 모델 훈련을 건너뜝니다.")
        return None

    def objective(trial):
        params = {
            'objective': 'regression_l1', # MAE
            'metric': 'rmse',
            'n_estimators': 10000,
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
            'num_leaves': trial.suggest_int('num_leaves', 31, 256),
            'max_depth': trial.suggest_int('max_depth', 7, 15),
            'min_child_samples': trial.suggest_int('min_child_samples', 20, 100),
            'subsample': trial.suggest_float('subsample', 0.7, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.7, 1.0),
            'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 10.0, log=True),
            'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True),
            'random_state': SEED,
            'n_jobs': -1,
            'verbose': -1,
        }

        cv_rmses = []
        for train_idx, val_idx in cv_splits:
            X_train, X_val = X.loc[train_idx], X.loc[val_idx] # .iloc 대신 .loc 사용
            y_train, y_val = y.loc[train_idx], y.loc[val_idx] # .iloc 대신 .loc 사용

            model = lgb.LGBMRegressor(**params)
            model.fit(X_train, y_train,
                      eval_set=[(X_val, y_val)],
                      eval_metric='rmse',
                      callbacks=[lgb.early_stopping(100, verbose=False)],
                      categorical_feature=[f for f in categorical_features if f in X_train.columns])
            preds = model.predict(X_val)
            cv_rmses.append(np.sqrt(mean_squared_error(y_val, preds)))

        return np.mean(cv_rmses)

    # study = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=SEED))
    # study.optimize(objective, n_trials=50, show_progress_bar=True) # n_trials은 시간/성능 고려하여 조절

    # n_trials를 20으로 줄이고, MedianPruner를 사용하여 성능이 좋지 않은 trial은 조기에 중단
    study = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=SEED),
                                pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=30))
    study.optimize(objective, n_trials=20, show_progress_bar=True) # n_trials를 20으로 줄였습니다.
    print(f"   {group_name} 그룹 최적 RMSE: {study.best_value:.4f}")
    print(f"   최적 하이퍼파라미터: {study.best_params}")

    # 최종 모델 훈련
    best_params = study.best_params
    best_params['n_estimators'] = 10000
    best_params['objective'] = 'regression_l1'
    best_params['metric'] = 'rmse'
    best_params['random_state'] = SEED

    final_model = lgb.LGBMRegressor(**best_params)

    # 단조성 제약 추가
    monotone_constraints = get_monotone_constraints(features)

    final_model.fit(X, y,
                    categorical_feature=categorical_features,
                    eval_set=[(X, y)], # 전체 데이터로 마지막 성능 확인
                    callbacks=[lgb.early_stopping(100), lgb.log_evaluation(1000)],
                    feature_name=features,
                    # monotone_constraints=monotone_constraints # 필요시 주석 해제
                   )

    # 모델 저장
    joblib.dump(final_model, f'{PATH}lgb_model_{group_name}.pkl')
    joblib.dump(features, f'{PATH}features_{group_name}.pkl')
    joblib.dump(categorical_features, f'{PATH}categorical_features_{group_name}.pkl')

    return final_model

# ---------------------------------------------------
# 3. 메인 실행 로직
# ---------------------------------------------------

if __name__ == "__main__":
    # 1. 데이터 로드 및 전체 전처리
    print("1. 데이터 로드 및 전처리 시작...")
    train_df = pd.read_csv(PATH + "train_heat.csv")
    test_df = pd.read_csv(PATH + "test_heat.csv")

    # 디버깅을 위해 test_df의 초기 컬럼 목록을 출력합니다.
    print(f"Test DataFrame 초기 컬럼: {test_df.columns.tolist()}")

    train_df = load_and_preprocess(train_df)
    test_df = load_and_preprocess(test_df)

    # train_df에만 'heat_demand' 컬럼이 있다고 가정하고 create_missing_flags_and_replace 함수 호출
    train_df = create_missing_flags_and_replace(train_df)

    # test_df의 'heat_demand'는 예측 대상이므로, create_missing_flags_and_replace 함수에서 처리되지 않도록 직접 처리
    # 대신, 'ta', 'ws' 등 기상 관련 컬럼의 -99 결측치만 처리
    temp_missing_cols_test = [col for col in ['ta', 'ws', 'rn_day', 'rn_hr1', 'hm', 'si', 'ta_chi'] if col in test_df.columns]
    for col in temp_missing_cols_test:
        missing_mask = (test_df[col] == -99)
        test_df[f'{col}_missing'] = missing_mask.astype(int)
        test_df[col] = test_df[col].replace(-99, np.nan)
    if 'wd' in test_df.columns:
        test_df['wd'] = test_df['wd'].replace(-9.9, np.nan)

    # 훈련 데이터에 대해서만 결측치 보간 (테스트는 훈련 세터의 통계량을 쓰지 않음)
    train_df = apply_interpolation(train_df)
    # 테스트 데이터는 자체적으로 보간
    test_df = apply_interpolation(test_df)

    train_df = add_heating_season(train_df)
    test_df = add_heating_season(test_df)

    # 이상치 플래그는 훈련 데이터 기준으로 생성하여 테스트에 적용
    train_df, test_df = create_weather_outlier_flags(train_df, test_df)

    train_df = create_advanced_features(train_df)
    test_df = create_advanced_features(test_df)

    print("전처리 완료!")

    # 2. 시즌별 그룹 분할
    train_groups = {name: data for name, data in train_df.groupby('heating_season')}
    test_groups = {name: data for name, data in test_df.groupby('heating_season')}

    # 3. 그룹별 모델 훈련 및 예측
    print("\n2. 시즌별 모델 훈련 시작...")
    models = {}
    for group_id, df_group in train_groups.items():
        group_name = 'heating' if group_id == 1 else 'non_heating'
        if len(df_group) > 0:
            trained_model = optimize_and_train(df_group, group_name)
            if trained_model is not None: # 모델 훈련이 성공적으로 완료된 경우에만 저장
                models[group_name] = trained_model

    # 4. 테스트 데이터 예측 및 통합
    print("\n3. 테스트 데이터 예측 및 제출 파일 생성...")
    # 원본 test_df의 인덱스를 기반으로 최종 예측을 저장할 Series 생성
    final_predictions = pd.Series(np.zeros(len(test_df)), index=test_df.index)

    for group_id, df_group in test_groups.items():
        group_name = 'heating' if group_id == 1 else 'non_heating'
        if len(df_group) > 0 and group_name in models:
            print(f"   {group_name} 그룹 예측 중...")
            model = models[group_name]

            # 훈련 시 사용된 피처 순서대로 테스트 데이터 준비
            # 모델을 로드하는 대신, 현재 메모리에 있는 모델을 사용하고 features도 이미 사용된 것을 활용합니다.
            # 하지만, 안전을 위해 파일로 저장된 features를 로드하는 기존 로직 유지
            try:
                train_features = joblib.load(f'{PATH}features_{group_name}.pkl')
            except FileNotFoundError:
                print(f"경고: {group_name} 그룹의 피처 파일이 없습니다. 예측을 건너뜝니다.")
                continue

            # test_df_group에 train_features에 없는 컬럼이 있다면 0으로 채우고,
            # train_features에 있는 컬럼이 test_df_group에 없다면 NaN이 됩니다.
            # 따라서 reindex를 사용하고 fill_value=0으로 설정
            X_test_group = df_group.reindex(columns=train_features, fill_value=0)

            # 예측 전에 컬럼 정합성 다시 확인 (선택 사항이지만 안전성 증가)
            if not all(f in X_test_group.columns for f in train_features):
                print(f"경고: {group_name} 그룹 예측 시 훈련 피처와 테스트 피처의 불일치 발생. 예측을 건너뜝니다.")
                continue

            preds = model.predict(X_test_group)
            final_predictions.loc[df_group.index] = preds
        else:
            print(f"   {group_name} 그룹에 대한 모델이 없거나 데이터가 없어 예측을 건너뜺니다.")

    # 예측값 후처리
    final_predictions = np.maximum(final_predictions, 0) # 0보다 작은 값은 0으로
    final_predictions = np.round(final_predictions, 1)

    # 제출 파일 생성
    # 원본 test_heat.csv를 다시 로드하여 'tm'과 'branch_id' 컬럼을 가져옵니다.
    # 이미 load_and_preprocess에서 컬럼명을 소문자로 변경했으므로, 다시 로드할 때는 원본 컬럼명을 사용할 수 있도록 주의
    original_test_submission = pd.read_csv(PATH + "test_heat.csv")
    submission = original_test_submission[['TM', 'branch_ID']].copy() # 원본 대소문자 컬럼명 사용

    # submission 데이터프레임의 'TM'과 'branch_ID'를 'tm'과 'branch_id'로 변경하여 test_df와 통일
    submission.columns = ['tm', 'branch_id']

    # 최종 예측값을 원래 test_df의 인덱스에 맞게 매핑
    # test_df의 'tm'을 기준으로 합칠 것이므로, test_df도 원래 test_heat.csv와 순서가 같다고 가정
    # 또는 test_df에 고유 ID가 있다면 더 확실
    # 여기서는 test_df의 인덱스와 final_predictions의 인덱스가 매칭된다고 가정합니다.
    submission['heat_demand'] = final_predictions.values

    output_filename = "submission_lgbm_upgraded.csv"
    submission.to_csv(f"{PATH}{output_filename}", index=False)

    print(f"\n🎉 예측 완료! 제출 파일 '{output_filename}'이(가) '{PATH}' 경로에 저장되었습니다.")

Mounted at /content/drive
1. 데이터 로드 및 전처리 시작...
Test DataFrame 초기 컬럼: ['TM', 'branch_ID', 'TA', 'WD', 'WS', 'RN_DAY', 'RN_HR1', 'HM', 'SI', 'ta_chi', 'heat_demand']
지사별 시계열 보간 적용 중...
지사별 시계열 보간 적용 중...
경고: 보간 후에도 166915개의 결측치가 남아있습니다. 0으로 채웁니다.
시즌별 기상 이상치 플래그 생성 중...
전처리 완료!

2. 시즌별 모델 훈련 시작...

===== NON_HEATING 그룹 모델 훈련 시작 =====
   Fold: Val Year=2021, Train size=139536, Val size=69768


[I 2025-06-24 12:13:08,979] A new study created in memory with name: no-name-c443ecc0-307e-4081-8b75-7efbf341489d


   Fold: Val Year=2022, Train size=139536, Val size=69768
   Fold: Val Year=2023, Train size=139536, Val size=69768


  0%|          | 0/20 [00:00<?, ?it/s]

[I 2025-06-24 12:24:31,620] Trial 0 finished with value: 9.477276950175998 and parameters: {'learning_rate': 0.023688639503640783, 'num_leaves': 245, 'max_depth': 13, 'min_child_samples': 68, 'subsample': 0.7468055921327309, 'colsample_bytree': 0.7467983561008608, 'reg_alpha': 0.0017073967431528124, 'reg_lambda': 2.9154431891537547}. Best is trial 0 with value: 9.477276950175998.
[I 2025-06-24 12:28:29,604] Trial 1 finished with value: 9.455694595753966 and parameters: {'learning_rate': 0.039913058785616795, 'num_leaves': 191, 'max_depth': 7, 'min_child_samples': 98, 'subsample': 0.9497327922401265, 'colsample_bytree': 0.7637017332034828, 'reg_alpha': 0.005337032762603957, 'reg_lambda': 0.00541524411940254}. Best is trial 1 with value: 9.455694595753966.
[I 2025-06-24 12:36:44,537] Trial 2 finished with value: 9.425165735740231 and parameters: {'learning_rate': 0.02014847788415866, 'num_leaves': 149, 'max_depth': 10, 'min_child_samples': 43, 'subsample': 0.8835558684167139, 'colsample_

[I 2025-06-24 14:15:12,037] A new study created in memory with name: no-name-bac16ee7-209b-4acb-8c5c-a821e14b54d8


   Fold: Val Year=2023, Train size=193325, Val size=96672


  0%|          | 0/20 [00:00<?, ?it/s]

[I 2025-06-24 14:22:41,161] Trial 0 finished with value: 22.222188440479798 and parameters: {'learning_rate': 0.023688639503640783, 'num_leaves': 245, 'max_depth': 13, 'min_child_samples': 68, 'subsample': 0.7468055921327309, 'colsample_bytree': 0.7467983561008608, 'reg_alpha': 0.0017073967431528124, 'reg_lambda': 2.9154431891537547}. Best is trial 0 with value: 22.222188440479798.
[I 2025-06-24 14:27:15,219] Trial 1 finished with value: 22.3583039904058 and parameters: {'learning_rate': 0.039913058785616795, 'num_leaves': 191, 'max_depth': 7, 'min_child_samples': 98, 'subsample': 0.9497327922401265, 'colsample_bytree': 0.7637017332034828, 'reg_alpha': 0.005337032762603957, 'reg_lambda': 0.00541524411940254}. Best is trial 0 with value: 22.222188440479798.
[I 2025-06-24 14:34:45,614] Trial 2 finished with value: 22.243135641896913 and parameters: {'learning_rate': 0.02014847788415866, 'num_leaves': 149, 'max_depth': 10, 'min_child_samples': 43, 'subsample': 0.8835558684167139, 'colsamp

In [ ]:
!pip install optuna-integration[lightgbm]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.9/98.9 kB 7.4 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import joblib
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
import optuna
from tqdm.auto import tqdm
import warnings
import holidays
import os

warnings.filterwarnings('ignore')

# Google Drive 마운트 (Colab 환경인 경우)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    PATH = "/content/drive/MyDrive/열수요예측/"
    # 디렉토리가 없으면 생성
    if not os.path.exists(PATH):
        os.makedirs(PATH)
except ImportError:
    PATH = "./" # 로컬 환경

# SEED 고정
SEED = 42
np.random.seed(SEED)

# ---------------------------------------------------
# 1. 데이터 전처리 및 피처 엔지니어링 (노트북 기반)
# ---------------------------------------------------
def load_and_preprocess(df):
    """기본 컬럼명 정리 및 타입 변환 (tm 컬럼 자동 탐지 기능 추가)"""
    df = df.copy()
    if 'Unnamed: 0' in df.columns:
        df = df.drop(columns=['Unnamed: 0'])

    # 모든 컬럼 이름을 소문자로 변경하여 대소문자 문제를 해결합니다.
    df.columns = [col.lower() for col in df.columns]

    # 'train_heat.' 또는 'test_heat.' 접두사 일괄 제거
    df.columns = [col.replace('train_heat.', '').replace('test_heat.', '') for col in df.columns]

    # 'tm' 컬럼이 없는 경우, 'tm'을 포함하는 다른 컬럼을 찾아 이름 변경
    if 'tm' not in df.columns:
        time_col = [col for col in df.columns if 'tm' in col]
        if time_col:
            print(f"'{time_col[0]}' 컬럼을 'tm' 컬럼으로 변경합니다.")
            df.rename(columns={time_col[0]: 'tm'}, inplace=True)
        else:
            print(f"현재 데이터프레임의 컬럼 목록: {df.columns.tolist()}")
            raise KeyError("데이터에서 'tm' 또는 'tm'을 포함하는 시간 관련 컬럼을 찾을 수 없습니다.")

    # tm 컬럼 타입 변환
    if df['tm'].dtype == 'object':
        df['tm'] = pd.to_datetime(df['tm'])
    else:
        df['tm'] = pd.to_datetime(df['tm'], format='%Y%m%d%H')

    # branch_id 컬럼을 category 타입으로 변환
    if 'branch_id' in df.columns:
        df['branch_id'] = df['branch_id'].astype('category')

    return df

def create_missing_flags_and_replace(df):
    """결측치 플래그 생성 및 NaN으로 변환"""
    df = df.copy()
    missing_cols = ['ta', 'ws', 'rn_day', 'rn_hr1', 'hm', 'si', 'ta_chi', 'heat_demand']

    # -99를 결측치로 인식하여 플래그 생성
    for col in missing_cols:
        if col in df.columns: # 컬럼이 존재하는지 확인
            missing_mask = (df[col] == -99)
            df[f'{col}_missing'] = missing_mask.astype(int)
            df[col] = df[col].replace(-99, np.nan)

    # wd 컬럼 처리
    if 'wd' in df.columns: # 컬럼이 존재하는지 확인
        df['wd'] = df['wd'].replace(-9.9, np.nan)

    return df

def apply_interpolation(df):
    """지사별 시계열 보간"""
    df = df.copy()
    print("지사별 시계열 보간 적용 중...")

    # 보간 대상 컬럼
    interp_cols = ['ta', 'ws', 'rn_day', 'rn_hr1', 'hm', 'si', 'ta_chi', 'wd']
    if 'heat_demand' in df.columns:
        interp_cols.append('heat_demand')

    # 실제 존재하는 보간 대상 컬럼만 필터링
    current_interp_cols = [col for col in interp_cols if col in df.columns]

    # tm 컬럼 기준으로 먼저 정렬 (보간의 정확성을 높임)
    df = df.sort_values(by=['branch_id', 'tm']).reset_index(drop=True)

    # 기존 df를 그대로 사용하면서 보간을 적용
    for col in current_interp_cols:
        df[col] = df.groupby('branch_id')[col].transform(lambda group: group.interpolate(method='linear', limit_direction='both'))

    # 보간 후에도 남은 결측치 확인 (NaN이 남아있다면)
    remaining_nan_before_fill = df[current_interp_cols].isnull().sum().sum()
    if remaining_nan_before_fill > 0:
        print(f"경고: 보간 후에도 {remaining_nan_before_fill}개의 결측치가 남아있습니다. 0으로 채웁니다.")

        # branch_id 컬럼은 fillna(0)에서 제외합니다.
        cols_to_fill_zero = [col for col in current_interp_cols if col != 'branch_id']
        df[cols_to_fill_zero] = df[cols_to_fill_zero].fillna(0)

    return df

def add_heating_season(df):
    """난방 시즌 컬럼 추가"""
    df = df.copy()
    df['month'] = df['tm'].dt.month
    df['heating_season'] = 0
    heating_months = [10, 11, 12, 1, 2, 3, 4]
    df.loc[df['month'].isin(heating_months), 'heating_season'] = 1
    return df

def create_advanced_features(df):
    """고도화된 파생변수 생성"""
    df = df.copy()

    # 시간 관련 변수
    df['year'] = df['tm'].dt.year
    df['day'] = df['tm'].dt.day
    df['hour'] = df['tm'].dt.hour
    df['dayofweek'] = df['tm'].dt.dayofweek
    df['dayofyear'] = df['tm'].dt.dayofyear

    # 순환 인코딩
    df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
    df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)
    df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
    df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)
    df['dayofweek_sin'] = np.sin(2 * np.pi * df['dayofweek'] / 7)
    df['dayofweek_cos'] = np.cos(2 * np.pi * df['dayofweek'] / 7)

    # 고급 기상 변수
    df['HDD18'] = np.maximum(0, 18 - df['ta']) # 난방도일

    # 체감온도 계산 (겨울용)
    def calculate_apparent_temp(ta, ws):
        return 13.12 + 0.6215 * ta - 11.37 * (ws * 3.6)**0.16 + 0.3965 * ta * (ws * 3.6)**0.16
    df['apparent_temp'] = calculate_apparent_temp(df['ta'], df['ws'])

    # 시차 및 이동평균 (지사별)
    for lag in [3, 6, 24]:
        if 'ta' in df.columns:
            df[f'ta_lag_{lag}h'] = df.groupby('branch_id')['ta'].shift(lag)
    for window in [6, 12, 24]:
        if 'ta' in df.columns:
            df[f'ta_ma_{window}h'] = df.groupby('branch_id')['ta'].transform(lambda x: x.rolling(window, min_periods=1).mean())
    if 'ta' in df.columns:
        df['ta_diff_3h'] = df.groupby('branch_id')['ta'].diff(3)
        df['ta_diff_6h'] = df.groupby('branch_id')['ta'].diff(6)

    # 공휴일
    kr_holidays = holidays.KR()
    df['is_holiday'] = df['tm'].dt.date.apply(lambda x: x in kr_holidays).astype(int)

    # 다시 보간 (시차 변수 생성 후 생긴 NaN 처리)
    # heat_demand가 없는 경우를 고려하여 drop할 컬럼 목록 동적 생성
    cols_to_drop = ['tm']
    if 'heat_demand' in df.columns:
        cols_to_drop.append('heat_demand')

    # 모든 컬럼을 대상으로 NaN을 처리하기 전에, drop할 컬럼만 제외
    feature_cols_for_fillna = [col for col in df.columns if col not in cols_to_drop]

    df[feature_cols_for_fillna] = df.groupby('branch_id')[feature_cols_for_fillna].transform(lambda x: x.ffill().bfill())

    # branch_id 컬럼을 제외하고 fillna(0) 적용
    cols_for_global_fillna = [col for col in df.columns if col not in cols_to_drop and df[col].dtype != 'category']
    df[cols_for_global_fillna] = df[cols_for_global_fillna].fillna(0)

    return df

def create_weather_outlier_flags(train_df, test_df):
    """시즌별 기상 이상치 플래그 생성 (훈련 세트 기준)"""
    print("시즌별 기상 이상치 플래그 생성 중...")
    outlier_thresholds = {}

    for branch in train_df['branch_id'].unique():
        branch_data = train_df[train_df['branch_id'] == branch]
        outlier_thresholds[branch] = {}
        for season in [0, 1]:
            season_data = branch_data[branch_data['heating_season'] == season]
            if len(season_data) > 10:
                outlier_thresholds[branch][season] = {
                    'ta_q10': season_data['ta'].quantile(0.10), # 극한 추위
                    'ws_q90': season_data['ws'].quantile(0.90), # 강풍
                    'rn_day_q90': season_data['rn_day'].quantile(0.90) # 폭우
                }

    def apply_thresholds(df, thresholds):
        df = df.copy()
        # 이상치 플래그 컬럼 초기화 (초기화 전에 이미 존재한다면 덮어씁니다)
        df['cold_extreme'] = 0
        df['strong_wind'] = 0
        df['heavy_rain'] = 0

        for branch in df['branch_id'].unique():
            if branch in thresholds:
                branch_mask = df['branch_id'] == branch
                for season in [0, 1]:
                    if season in thresholds[branch]:
                        season_mask = branch_mask & (df['heating_season'] == season)
                        season_thresholds = thresholds[branch][season]

                        # 해당 컬럼이 데이터프레임에 있는지 확인 후 적용
                        if 'ta' in df.columns:
                            df.loc[season_mask, 'cold_extreme'] = (df.loc[season_mask, 'ta'] < season_thresholds['ta_q10']).astype(int)
                        if 'ws' in df.columns:
                            df.loc[season_mask, 'strong_wind'] = (df.loc[season_mask, 'ws'] > season_thresholds['ws_q90']).astype(int)
                        if 'rn_day' in df.columns:
                            df.loc[season_mask, 'heavy_rain'] = (df.loc[season_mask, 'rn_day'] > season_thresholds['rn_day_q90']).astype(int)
        return df

    train_result = apply_thresholds(train_df, outlier_thresholds)
    test_result = apply_thresholds(test_df, outlier_thresholds)

    return train_result, test_result

def get_feature_names(df):
    """사용할 피처 이름 정의"""
    # 원본 숫자/범주 + 결측치 플래그 + 이상치 플래그 + 파생변수
    # df.columns는 이미 소문자로 되어있을 것으로 가정
    features = [col for col in df.columns if col not in ['tm', 'heat_demand', 'year']]

    # 범주형 피처 명시
    categorical_features = [
        'branch_id', 'month', 'hour', 'dayofweek', 'heating_season', 'is_holiday',
        'cold_extreme', 'strong_wind', 'heavy_rain'
    ]
    # 실제 존재하는 범주형 피처만 필터링
    categorical_features = [f for f in categorical_features if f in df.columns]

    return features, categorical_features

# ---------------------------------------------------
# 2. 모델 훈련 및 최적화
# ---------------------------------------------------

def create_year_based_cv_splits(df):
    """연도 기반 3-Fold CV 분할 생성"""
    cv_splits = []
    years = sorted(df['year'].unique())

    # 마지막 3개년을 검증 세트로 사용
    val_years_to_use = years[-3:] if len(years) >= 3 else years

    if not val_years_to_use: # val_years_to_use가 비어있는 경우
        print("경고: 연도 데이터가 충분하지 않아 연도 기반 CV를 생성할 수 없습니다. 일반 KFold(3-Fold)를 사용합니다.")
        kf = KFold(n_splits=3, shuffle=True, random_state=SEED)
        cv_splits = list(kf.split(df))
        return cv_splits

    for val_year in val_years_to_use:
        train_mask = (df['year'] != val_year)
        val_mask = (df['year'] == val_year)

        train_indices = df[train_mask].index.tolist()
        val_indices = df[val_mask].index.tolist()

        # 데이터가 있는 경우에만 추가
        if len(train_indices) > 0 and len(val_indices) > 0:
            cv_splits.append((train_indices, val_indices))
            print(f"   Fold: Val Year={val_year}, Train size={len(train_indices)}, Val size={len(val_indices)}")

    if not cv_splits:
        # 데이터가 너무 적을 경우 일반 KFold로 대체 (이미 위에서 처리되었지만, 안전을 위해 남겨둠)
        print("연도 기반 CV를 생성할 데이터가 부족하여 일반 KFold(3-Fold)를 사용합니다.")
        kf = KFold(n_splits=3, shuffle=True, random_state=SEED)
        cv_splits = list(kf.split(df))

    return cv_splits

def get_monotone_constraints(feature_names):
    """LightGBM 단조성 제약 설정"""
    constraints = [0] * len(feature_names)
    for i, feature in enumerate(feature_names):
        if 'ta' in feature or 'apparent_temp' in feature:
            constraints[i] = -1 # 온도/체감온도 상승 시 열수요 감소
        elif feature == 'HDD18':
            constraints[i] = 1 # 난방도일 상승 시 열수요 증가
        elif 'ws' in feature:
            constraints[i] = 1 # 풍속 상승 시 열수요 증가
    return constraints

def optimize_and_train(df_group, group_name):
    """Optuna로 하이퍼파라미터 최적화 및 모델 훈련"""
    print(f"\n===== {group_name.upper()} 그룹 모델 훈련 시작 =====")

    # heat_demand 컬럼이 없는 경우 (테스트 데이터에서 호출 시) 예외 처리
    if 'heat_demand' not in df_group.columns:
        print(f"경고: {group_name} 그룹 데이터에 'heat_demand' 컬럼이 없습니다. 모델 훈련을 건너뜁니다.")
        return None

    X = df_group.drop(columns=['tm', 'heat_demand'])
    y = df_group['heat_demand']
    features, categorical_features = get_feature_names(X)
    X = X[features] # 정의된 features만 사용

    # X에 branch_id가 없으면 문제가 되므로 확인
    if 'branch_id' not in X.columns:
        print(f"오류: {group_name} 그룹 데이터에 'branch_id' 컬럼이 없습니다. 모델 훈련 중단.")
        return None

    # 연도 기반 CV 분할
    cv_splits = create_year_based_cv_splits(df_group)

    # CV 분할이 유효한지 확인
    if not cv_splits:
        print(f"경고: {group_name} 그룹에 유효한 CV 분할이 없습니다. 모델 훈련을 건너뭅니다.")
        return None

    def objective(trial):
        params = {
            'objective': 'regression_l1', # MAE
            'metric': 'rmse',
            'n_estimators': 2000, # 10000 -> 2000으로 줄였습니다. (early stopping에 의해 조절됨)
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
            'num_leaves': trial.suggest_int('num_leaves', 31, 256),
            'max_depth': trial.suggest_int('max_depth', 7, 15),
            'min_child_samples': trial.suggest_int('min_child_samples', 20, 100),
            'subsample': trial.suggest_float('subsample', 0.7, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.7, 1.0),
            'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 10.0, log=True),
            'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True),
            'random_state': SEED,
            'n_jobs': -1,
            'verbose': -1,
        }

        cv_rmses = []
        # Optuna 프루닝 콜백 추가
        pruning_callback = optuna.integration.LightGBMPruningCallback(trial, "rmse")

        for train_idx, val_idx in cv_splits:
            X_train, X_val = X.loc[train_idx], X.loc[val_idx]
            y_train, y_val = y.loc[train_idx], y.loc[val_idx]

            model = lgb.LGBMRegressor(**params)
            model.fit(X_train, y_train,
                      eval_set=[(X_val, y_val)],
                      eval_metric='rmse',
                      callbacks=[lgb.early_stopping(100, verbose=False), pruning_callback], # 프루닝 콜백 적용
                      categorical_feature=[f for f in categorical_features if f in X_train.columns])
            preds = model.predict(X_val)
            cv_rmses.append(np.sqrt(mean_squared_error(y_val, preds)))

        return np.mean(cv_rmses)

    # n_trials를 20으로 줄이고, MedianPruner를 사용하여 성능이 좋지 않은 trial은 조기에 중단
    study = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=SEED),
                                pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=30))
    study.optimize(objective, n_trials=20, show_progress_bar=True) # n_trials를 20으로 줄였습니다.

    print(f"   {group_name} 그룹 최적 RMSE: {study.best_value:.4f}")
    print(f"   최적 하이퍼파라미터: {study.best_params}")

    # 최종 모델 훈련
    best_params = study.best_params
    best_params['n_estimators'] = 10000 # 최종 모델은 충분히 학습하도록 다시 10000으로
    best_params['objective'] = 'regression_l1'
    best_params['metric'] = 'rmse'
    best_params['random_state'] = SEED

    final_model = lgb.LGBMRegressor(**best_params)

    # 단조성 제약 추가
    monotone_constraints = get_monotone_constraints(features)

    final_model.fit(X, y,
                    categorical_feature=categorical_features,
                    eval_set=[(X, y)], # 전체 데이터로 마지막 성능 확인
                    callbacks=[lgb.early_stopping(100), lgb.log_evaluation(1000)],
                    feature_name=features,
                    # monotone_constraints=monotone_constraints # 필요시 주석 해제
                   )

    # 모델 저장
    joblib.dump(final_model, f'{PATH}lgb_model_{group_name}.pkl')
    joblib.dump(features, f'{PATH}features_{group_name}.pkl')
    joblib.dump(categorical_features, f'{PATH}categorical_features_{group_name}.pkl')

    return final_model

# ---------------------------------------------------
# 3. 메인 실행 로직
# ---------------------------------------------------

if __name__ == "__main__":
    # 1. 데이터 로드 및 전체 전처리
    print("1. 데이터 로드 및 전처리 시작...")
    train_df = pd.read_csv(PATH + "train_heat.csv")
    test_df = pd.read_csv(PATH + "test_heat.csv")

    # 디버깅을 위해 test_df의 초기 컬럼 목록을 출력합니다.
    print(f"Test DataFrame 초기 컬럼: {test_df.columns.tolist()}")

    train_df = load_and_preprocess(train_df)
    test_df = load_and_preprocess(test_df)

    # train_df에만 'heat_demand' 컬럼이 있다고 가정하고 create_missing_flags_and_replace 함수 호출
    train_df = create_missing_flags_and_replace(train_df)

    # test_df의 'heat_demand'는 예측 대상이므로, create_missing_flags_and_replace 함수에서 처리되지 않도록 직접 처리
    # 대신, 'ta', 'ws' 등 기상 관련 컬럼의 -99 결측치만 처리
    temp_missing_cols_test = [col for col in ['ta', 'ws', 'rn_day', 'rn_hr1', 'hm', 'si', 'ta_chi'] if col in test_df.columns]
    for col in temp_missing_cols_test:
        missing_mask = (test_df[col] == -99)
        test_df[f'{col}_missing'] = missing_mask.astype(int)
        test_df[col] = test_df[col].replace(-99, np.nan)
    if 'wd' in test_df.columns:
        test_df['wd'] = test_df['wd'].replace(-9.9, np.nan)

    # 훈련 데이터에 대해서만 결측치 보간 (테스트는 훈련 세터의 통계량을 쓰지 않음)
    train_df = apply_interpolation(train_df)
    # 테스트 데이터는 자체적으로 보간
    test_df = apply_interpolation(test_df)

    train_df = add_heating_season(train_df)
    test_df = add_heating_season(test_df)

    # 이상치 플래그는 훈련 데이터 기준으로 생성하여 테스트에 적용
    train_df, test_df = create_weather_outlier_flags(train_df, test_df)

    train_df = create_advanced_features(train_df)
    test_df = create_advanced_features(test_df)

    print("전처리 완료!")

    # 2. 시즌별 그룹 분할
    train_groups = {name: data for name, data in train_df.groupby('heating_season')}
    test_groups = {name: data for name, data in test_df.groupby('heating_season')}

    # 3. 그룹별 모델 훈련 및 예측
    print("\n2. 시즌별 모델 훈련 시작...")
    models = {}
    for group_id, df_group in train_groups.items():
        group_name = 'heating' if group_id == 1 else 'non_heating'
        if len(df_group) > 0:
            trained_model = optimize_and_train(df_group, group_name)
            if trained_model is not None: # 모델 훈련이 성공적으로 완료된 경우에만 저장
                models[group_name] = trained_model

    # 4. 테스트 데이터 예측 및 통합
    print("\n3. 테스트 데이터 예측 및 제출 파일 생성...")
    # 원본 test_df의 인덱스를 기반으로 최종 예측을 저장할 Series 생성
    final_predictions = pd.Series(np.zeros(len(test_df)), index=test_df.index)

    for group_id, df_group in test_groups.items():
        group_name = 'heating' if group_id == 1 else 'non_heating'
        if len(df_group) > 0 and group_name in models:
            print(f"   {group_name} 그룹 예측 중...")
            model = models[group_name]

            try:
                train_features = joblib.load(f'{PATH}features_{group_name}.pkl')
            except FileNotFoundError:
                print(f"경고: {group_name} 그룹의 피처 파일이 없습니다. 예측을 건너뭅니다.")
                continue

            X_test_group = df_group.reindex(columns=train_features, fill_value=0)

            if not all(f in X_test_group.columns for f in train_features):
                print(f"경고: {group_name} 그룹 예측 시 훈련 피처와 테스트 피처의 불일치 발생. 예측을 건너뭅니다.")
                continue

            preds = model.predict(X_test_group)
            final_predictions.loc[df_group.index] = preds
        else:
            print(f"   {group_name} 그룹에 대한 모델이 없거나 데이터가 없어 예측을 건너뜺니다.")

    # 예측값 후처리
    final_predictions = np.maximum(final_predictions, 0) # 0보다 작은 값은 0으로
    final_predictions = np.round(final_predictions, 1)

    # 제출 파일 생성
    original_test_submission = pd.read_csv(PATH + "test_heat.csv")
    submission = original_test_submission[['TM', 'branch_ID']].copy() # 원본 대소문자 컬럼명 사용

    submission.columns = ['tm', 'branch_id']

    submission['heat_demand'] = final_predictions.values

    output_filename = "submission_lgbm_upgraded_faster.csv" # 파일명 변경
    submission.to_csv(f"{PATH}{output_filename}", index=False)

    print(f"\n🎉 예측 완료! 제출 파일 '{output_filename}'이(가) '{PATH}' 경로에 저장되었습니다.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
1. 데이터 로드 및 전처리 시작...
Test DataFrame 초기 컬럼: ['TM', 'branch_ID', 'TA', 'WD', 'WS', 'RN_DAY', 'RN_HR1', 'HM', 'SI', 'ta_chi', 'heat_demand']
지사별 시계열 보간 적용 중...
지사별 시계열 보간 적용 중...
경고: 보간 후에도 166915개의 결측치가 남아있습니다. 0으로 채웁니다.
시즌별 기상 이상치 플래그 생성 중...
전처리 완료!

2. 시즌별 모델 훈련 시작...

===== NON_HEATING 그룹 모델 훈련 시작 =====
   Fold: Val Year=2021, Train size=139536, Val size=69768
   Fold: Val Year=2022, Train size=139536, Val size=69768


[I 2025-06-23 07:01:23,460] A new study created in memory with name: no-name-effb93b6-c61c-4066-91cc-f4a260eedb80


   Fold: Val Year=2023, Train size=139536, Val size=69768


  0%|          | 0/20 [00:00<?, ?it/s]

[I 2025-06-23 07:09:08,977] Trial 0 finished with value: 9.487658618560035 and parameters: {'learning_rate': 0.023688639503640783, 'num_leaves': 245, 'max_depth': 13, 'min_child_samples': 68, 'subsample': 0.7468055921327309, 'colsample_bytree': 0.7467983561008608, 'reg_alpha': 0.0017073967431528124, 'reg_lambda': 2.9154431891537547}. Best is trial 0 with value: 9.487658618560035.
[I 2025-06-23 07:13:03,581] Trial 1 finished with value: 9.456573981820746 and parameters: {'learning_rate': 0.039913058785616795, 'num_leaves': 191, 'max_depth': 7, 'min_child_samples': 98, 'subsample': 0.9497327922401265, 'colsample_bytree': 0.7637017332034828, 'reg_alpha': 0.005337032762603957, 'reg_lambda': 0.00541524411940254}. Best is trial 1 with value: 9.456573981820746.
[I 2025-06-23 07:18:54,369] Trial 2 finished with value: 9.442042659190664 and parameters: {'learning_rate': 0.02014847788415866, 'num_leaves': 149, 'max_depth': 10, 'min_child_samples': 43, 'subsample': 0.8835558684167139, 'colsample_

[I 2025-06-23 07:57:07,341] A new study created in memory with name: no-name-8945005c-4c27-423f-b453-e619706aaa71


   Fold: Val Year=2022, Train size=193325, Val size=96672
   Fold: Val Year=2023, Train size=193325, Val size=96672


  0%|          | 0/20 [00:00<?, ?it/s]

[I 2025-06-23 08:04:02,412] Trial 0 finished with value: 22.232134547767668 and parameters: {'learning_rate': 0.023688639503640783, 'num_leaves': 245, 'max_depth': 13, 'min_child_samples': 68, 'subsample': 0.7468055921327309, 'colsample_bytree': 0.7467983561008608, 'reg_alpha': 0.0017073967431528124, 'reg_lambda': 2.9154431891537547}. Best is trial 0 with value: 22.232134547767668.
[I 2025-06-23 08:08:22,221] Trial 1 finished with value: 22.369462329040704 and parameters: {'learning_rate': 0.039913058785616795, 'num_leaves': 191, 'max_depth': 7, 'min_child_samples': 98, 'subsample': 0.9497327922401265, 'colsample_bytree': 0.7637017332034828, 'reg_alpha': 0.005337032762603957, 'reg_lambda': 0.00541524411940254}. Best is trial 0 with value: 22.232134547767668.
[I 2025-06-23 08:13:31,286] Trial 2 finished with value: 22.28044461801785 and parameters: {'learning_rate': 0.02014847788415866, 'num_leaves': 149, 'max_depth': 10, 'min_child_samples': 43, 'subsample': 0.8835558684167139, 'colsam

# 마지막 실험

In [ ]:
!pip install pandas numpy scikit-learn catboost prophet optuna holidays tqdm joblib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.2/99.2 MB 5.6 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np
import joblib
import pickle
import os
import warnings
from datetime import datetime
import holidays
from tqdm.auto import tqdm
import random

# 머신러닝 라이브러리
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge

# 모델링 라이브러리
import catboost as cb
from catboost import CatBoostRegressor
from prophet import Prophet
import optuna
from optuna.samplers import TPESampler

# 경고 메시지 무시
warnings.filterwarnings('ignore')
# Prophet/Stan 로그 최소화
import logging
logging.getLogger('cmdstanpy').setLevel(logging.WARNING)
logging.getLogger('prophet').setLevel(logging.CRITICAL)


# Google Drive 마운트 (Colab 환경인 경우)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    PATH = "/content/drive/MyDrive/열수요예측/"
    # 디렉토리가 없으면 생성
    if not os.path.exists(PATH):
        os.makedirs(PATH)
except ImportError:
    PATH = "./" # 로컬 환경

# SEED 고정
SEED = 42
np.random.seed(SEED)
random.seed(SEED)

# ---------------------------------------------------
# 1. 데이터 전처리 및 피처 엔지니어링 (Notebook 기반 고도화)
# ---------------------------------------------------

def load_and_preprocess_advanced(train_path, test_path):
    """(Notebook) 고도화된 데이터 로드 및 전처리 (결측치 플래그 생성)"""
    print("고도화된 데이터 로드 및 전처리...")
    train_df = pd.read_csv(train_path)
    test_df = pd.read_csv(test_path)

    def process_df(df):
        if 'Unnamed: 0' in df.columns:
            df = df.drop(columns=['Unnamed: 0'])
        df.columns = [col.lower().replace('train_heat.', '').replace('test_heat.', '') for col in df.columns]

        if df['tm'].dtype == 'object':
            df['tm'] = pd.to_datetime(df['tm'])
        else:
            df['tm'] = pd.to_datetime(df['tm'], format='%Y%m%d%H')

        df['year'] = df['tm'].dt.year
        df['month'] = df['tm'].dt.month
        df['day'] = df['tm'].dt.day
        df['hour'] = df['tm'].dt.hour
        df['dayofweek'] = df['tm'].dt.dayofweek
        df['dayofyear'] = df['tm'].dt.dayofyear

        missing_cols = ['ta', 'ws', 'rn_day', 'rn_hr1', 'hm', 'si', 'ta_chi', 'heat_demand']
        for col in missing_cols:
            if col in df.columns:
                df[f'{col}_missing'] = (df[col] == -99).astype(int)
                df[col] = df[col].replace(-99, np.nan)

        if 'wd' in df.columns:
            df['wd'] = df['wd'].replace(-9.9, np.nan)

        if 'si' in df.columns:
            night_mask = (df['hour'] < 8) | (df['hour'] > 18)
            df.loc[night_mask & df['si'].isna(), 'si'] = 0

        df = df.sort_values(['branch_id', 'tm'])
        return df

    train_df = process_df(train_df)
    test_df = process_df(test_df)
    return train_df, test_df

def apply_svr_interpolation(df, is_train=True):
    """(Notebook) SVR 기반 고급 보간"""
    print(f"SVR 보간 적용 중 ({'TRAIN' if is_train else 'TEST'} 데이터)...")
    df_interpolated = df.copy()
    base_cols = ['ta', 'hm', 'ws', 'wd', 'rn_day', 'rn_hr1', 'si', 'ta_chi']
    interpolation_cols = base_cols + ['heat_demand'] if is_train else base_cols
    available_cols = [col for col in interpolation_cols if col in df.columns]

    df_interpolated['day_of_year'] = df_interpolated['tm'].dt.dayofyear

    for branch in tqdm(df['branch_id'].unique(), desc="지사별 SVR 보간"):
        branch_mask = df_interpolated['branch_id'] == branch
        branch_data = df_interpolated[branch_mask].copy().sort_values('tm')

        if len(branch_data) < 24:
            print(f"경고: 지사 {branch} 데이터 부족({len(branch_data)}개), 선형 보간으로 대체합니다.")
            df_interpolated.loc[branch_mask] = branch_data.interpolate(method='linear', limit_direction='both').ffill().bfill()
            continue

        for col in available_cols:
            if branch_data[col].isna().sum() == 0: continue

            try:
                train_mask = ~branch_data[col].isna()
                if train_mask.sum() < 10: raise ValueError("훈련 데이터 부족")

                feature_cols = ['hour', 'day_of_year', 'month', 'dayofweek']
                X_train = branch_data.loc[train_mask, feature_cols].values
                y_train = branch_data.loc[train_mask, col].values
                X_pred = branch_data.loc[~train_mask, feature_cols].values

                if len(X_pred) == 0: continue

                scaler_X = StandardScaler().fit(X_train)
                scaler_y = StandardScaler().fit(y_train.reshape(-1, 1))

                X_train_scaled = scaler_X.transform(X_train)
                y_train_scaled = scaler_y.transform(y_train.reshape(-1, 1)).flatten()

                svr = SVR(kernel='rbf', C=1.0, gamma='scale')
                svr.fit(X_train_scaled, y_train_scaled)

                y_pred_scaled = svr.predict(scaler_X.transform(X_pred))
                y_pred = scaler_y.inverse_transform(y_pred_scaled.reshape(-1, 1)).flatten()

                branch_data.loc[~train_mask, col] = y_pred
                df_interpolated.loc[branch_mask] = branch_data

            except Exception as e:
                print(f"SVR 보간 실패 (지사 {branch}, 컬럼 {col}): {e}. 선형 보간으로 대체.")
                df_interpolated.loc[branch_mask] = branch_data.interpolate(method='linear', limit_direction='both').ffill().bfill()

    # 최종 결측치 ffill/bfill
    df_interpolated = df_interpolated.ffill().bfill().fillna(0)
    return df_interpolated

def create_weather_outlier_flags(train_df, test_df):
    """(Notebook) 시즌별 기상 이상치 플래그 (훈련 세트 기준)"""
    print("시즌별 기상 이상치 플래그 생성 중...")
    outlier_thresholds = {}
    train_df['heating_season'] = train_df['month'].isin([10,11,12,1,2,3,4]).astype(int)
    test_df['heating_season'] = test_df['month'].isin([10,11,12,1,2,3,4]).astype(int)

    for branch in train_df['branch_id'].unique():
        branch_data = train_df[train_df['branch_id'] == branch]
        outlier_thresholds[branch] = {}
        for season in [0, 1]:
            season_data = branch_data[branch_data['heating_season'] == season]
            if len(season_data) > 10:
                outlier_thresholds[branch][season] = {
                    'ta_q10': season_data['ta'].quantile(0.10),
                    'ws_q90': season_data['ws'].quantile(0.90),
                    'rn_day_q90': season_data['rn_day'].quantile(0.90)
                }
    def apply_thresholds(df, thresholds):
        df = df.copy()
        df['cold_extreme'] = 0; df['strong_wind'] = 0; df['heavy_rain'] = 0
        for branch in df['branch_id'].unique():
            if branch in thresholds:
                for season in [0, 1]:
                    if season in thresholds[branch]:
                        mask = (df['branch_id'] == branch) & (df['heating_season'] == season)
                        df.loc[mask, 'cold_extreme'] = (df.loc[mask, 'ta'] < thresholds[branch][season]['ta_q10']).astype(int)
                        df.loc[mask, 'strong_wind'] = (df.loc[mask, 'ws'] > thresholds[branch][season]['ws_q90']).astype(int)
                        df.loc[mask, 'heavy_rain'] = (df.loc[mask, 'rn_day'] > thresholds[branch][season]['rn_day_q90']).astype(int)
        return df
    return apply_thresholds(train_df, outlier_thresholds), apply_thresholds(test_df, outlier_thresholds)

def create_advanced_features(df, season_type="heating"):
    """(Notebook) 시즌별 고도화된 특성 생성"""
    df = df.copy()

    # 시간 변수
    df['hour_cat'] = df['hour'].astype(str)
    df['month_cat'] = df['month'].astype(str)
    df['weekday_name'] = df['dayofweek'].map({0:'Mon',1:'Tue',2:'Wed',3:'Thu',4:'Fri',5:'Sat',6:'Sun'}).astype(str)

    # 순환 인코딩
    df['hour_sin'] = np.sin(2 * np.pi * df['hour']/24); df['hour_cos'] = np.cos(2 * np.pi * df['hour']/24)
    df['month_sin'] = np.sin(2 * np.pi * df['month']/12); df['month_cos'] = np.cos(2 * np.pi * df['month']/12)
    df['dayofweek_sin'] = np.sin(2 * np.pi * df['dayofweek']/7); df['dayofweek_cos'] = np.cos(2 * np.pi * df['dayofweek']/7)

    # 시즌별 월 순서
    if season_type == "heating":
        df['heating_month_order'] = df['month'].map({10:0, 11:1, 12:2, 1:3, 2:4, 3:5, 4:6})
        df['heating_month_sin'] = np.sin(2*np.pi*df['heating_month_order']/7); df['heating_month_cos'] = np.cos(2*np.pi*df['heating_month_order']/7)
    else:
        df['non_heating_month_order'] = df['month'].map({5:0, 6:1, 7:2, 8:3, 9:4})
        df['non_heating_month_sin'] = np.sin(2*np.pi*df['non_heating_month_order']/5); df['non_heating_month_cos'] = np.cos(2*np.pi*df['non_heating_month_order']/5)

    df['branch_id'] = df['branch_id'].astype(str)

    # 공휴일
    kr_holidays = holidays.KR()
    df['is_holiday'] = df['tm'].dt.date.apply(lambda x: x in kr_holidays); df['holiday_type'] = df['is_holiday'].map({False:'Weekday', True:'Holiday'}).astype(str)

    # 고급 수치형
    df['HDD18'] = np.maximum(0, 18 - df['ta'])
    if season_type == "heating":
        df['apparent_temp'] = 13.12 + 0.6215*df['ta'] - 11.37*(df['ws']*3.6)**0.16 + 0.3965*df['ta']*(df['ws']*3.6)**0.16

    # 시차/이동평균
    for lag in [3, 6, 24]: df[f'ta_lag_{lag}h'] = df.groupby('branch_id')['ta'].shift(lag)
    for window in [6, 12, 24]: df[f'ta_ma_{window}h'] = df.groupby('branch_id')['ta'].transform(lambda x: x.rolling(window, min_periods=1).mean())
    df['ta_diff_3h'] = df.groupby('branch_id')['ta'].diff(3); df['ta_diff_6h'] = df.groupby('branch_id')['ta'].diff(6)

    # 일별 통계
    daily_stats = df.groupby(['branch_id', df['tm'].dt.date])['ta'].agg(['min','max','mean']).round(2)
    daily_stats.columns = ['daily_ta_min','daily_ta_max','daily_ta_mean']
    daily_stats['daily_temp_range'] = daily_stats['daily_ta_max'] - daily_stats['daily_ta_min']
    df = df.merge(daily_stats, left_on=['branch_id', df['tm'].dt.date], right_index=True, how='left')

    # 최종 결측치 처리
    df.fillna(0, inplace=True)
    return df

# ---------------------------------------------------
# 2. 모델별 피처 정의 (Notebook 기반)
# ---------------------------------------------------
def define_model_features():
    prophet_features = {'basic': ['ta', 'hm', 'ws', 'HDD18'], 'seasonal': ['hour_sin','hour_cos','dayofweek_sin','dayofweek_cos','ta_lag_3h','ta_lag_6h']}
    catboost_features = {
        'numerical': ['day','dayofyear','ta','hm','ws','rn_day','rn_hr1','si','ta_chi','HDD18','hour_sin','hour_cos','dayofweek_sin','dayofweek_cos','ta_lag_3h','ta_lag_6h','ta_lag_24h','ta_ma_6h','ta_ma_12h','ta_ma_24h','ta_diff_3h','ta_diff_6h','daily_ta_min','daily_ta_max','daily_ta_mean','daily_temp_range'],
        'categorical': ['branch_id','hour_cat','month_cat','weekday_name','holiday_type'],
        'flags': ['ta_missing','ws_missing','rn_day_missing','rn_hr1_missing','hm_missing','si_missing','ta_chi_missing','cold_extreme','strong_wind','heavy_rain']
    }
    prophet_heating = {**prophet_features, 'basic': prophet_features['basic'] + ['apparent_temp'], 'seasonal': prophet_features['seasonal'] + ['heating_month_order']}
    prophet_non_heating = {**prophet_features, 'seasonal': prophet_features['seasonal'] + ['non_heating_month_order']}
    catboost_heating = {**catboost_features, 'seasonal': ['apparent_temp','heating_month_order','heating_month_sin','heating_month_cos']}
    catboost_non_heating = {**catboost_features, 'seasonal': ['non_heating_month_order','non_heating_month_sin','non_heating_month_cos']}
    return {'prophet_heating': prophet_heating, 'prophet_non_heating': prophet_non_heating, 'catboost_heating': catboost_heating, 'catboost_non_heating': catboost_non_heating}

MODEL_FEATURES = define_model_features()

# ---------------------------------------------------
# 3. 모델 클래스 (Notebook 기반)
# ---------------------------------------------------
class ProphetOptimizedModel:
    def __init__(self, season_type="heating"):
        self.models = {}
        self.best_params = {}
        self.season_type = season_type
        self.regressors = MODEL_FEATURES[f'prophet_{self.season_type}']['basic'] + MODEL_FEATURES[f'prophet_{self.season_type}']['seasonal']

    def fit(self, df, target_col='heat_demand'):
        for branch in tqdm(df['branch_id'].unique(), desc=f"Prophet {self.season_type} 훈련", leave=False):
            branch_data = df[df['branch_id'] == branch].copy()
            if len(branch_data) < 50: continue

            prophet_df = pd.DataFrame({'ds': pd.to_datetime(branch_data['tm']), 'y': branch_data[target_col]})
            for reg in self.regressors:
                if reg in branch_data.columns: prophet_df[reg] = branch_data[reg].values

            model = Prophet(**self.best_params, daily_seasonality=True, weekly_seasonality=True, yearly_seasonality=True)
            for reg in self.regressors:
                if reg in prophet_df.columns: model.add_regressor(reg)

            model.fit(prophet_df)
            self.models[branch] = model

    def predict(self, df):
        predictions = []
        for branch in df['branch_id'].unique():
            if branch not in self.models:
                predictions.extend([0] * len(df[df['branch_id'] == branch]))
                continue

            branch_data = df[df['branch_id'] == branch].copy()
            future_df = pd.DataFrame({'ds': pd.to_datetime(branch_data['tm'])})
            for reg in self.regressors:
                if reg in branch_data.columns: future_df[reg] = branch_data[reg].values

            forecast = self.models[branch].predict(future_df)
            predictions.extend(np.maximum(forecast['yhat'].values, 0))
        return np.array(predictions)

class CatBoostOptimizedModel:
    def __init__(self, season_type="heating"):
        self.model = None
        self.best_params = {}
        self.season_type = season_type
        features_config = MODEL_FEATURES[f'catboost_{self.season_type}']
        self.feature_cols = features_config['numerical'] + features_config['categorical'] + features_config['flags'] + features_config['seasonal']
        self.categorical_features = features_config['categorical']

    def _prepare_data(self, df):
        df_prepared = df.copy()
        for col in self.categorical_features:
            if col in df_prepared.columns: df_prepared[col] = df_prepared[col].astype(str)

        # 훈련 시점의 컬럼 순서와 동일하게 맞춰줌
        available_features = [f for f in self.feature_cols if f in df_prepared.columns]
        return df_prepared[available_features]

    def fit(self, df, target_col='heat_demand'):
        X = self._prepare_data(df)
        y = df[target_col]
        self.model = CatBoostRegressor(**self.best_params, cat_features=self.categorical_features, random_seed=SEED, verbose=0, allow_writing_files=False)
        self.model.fit(X, y)

    def predict(self, df):
        X = self._prepare_data(df)
        return np.maximum(self.model.predict(X), 0)

# ---------------------------------------------------
# 4. 스태킹 앙상블 클래스 (Notebook 기반)
# ---------------------------------------------------
class AdvancedStackingEnsemble:
    def __init__(self, season_type="heating", group_name=""):
        self.season_type = season_type
        self.group_name = group_name
        self.models = {'prophet': ProphetOptimizedModel(season_type), 'catboost': CatBoostOptimizedModel(season_type)}
        self.meta_model = None
        self.predefined_params = self._get_predefined_params()

    def _get_predefined_params(self):
        if self.season_type == "heating":
            return {
                'prophet': {'changepoint_prior_scale': 0.0026, 'seasonality_prior_scale': 0.13, 'holidays_prior_scale': 5.4, 'seasonality_mode': 'multiplicative'},
                'catboost': {'iterations': 1678, 'depth': 5, 'learning_rate': 0.057, 'l2_leaf_reg': 12.25, 'border_count': 42, 'loss_function':'RMSE'},
                'ridge': {'alpha': 63.5}
            }
        else: # non_heating
            return {
                'prophet': {'changepoint_prior_scale': 0.001, 'seasonality_prior_scale': 3.65, 'holidays_prior_scale': 0.45, 'seasonality_mode': 'additive'},
                'catboost': {'iterations': 1818, 'depth': 8, 'learning_rate': 0.085, 'l2_leaf_reg': 14.95, 'border_count': 35, 'loss_function':'RMSE'},
                'ridge': {'alpha': 63.5}
            }

    def fit(self, train_df, cv_splits, target_col='heat_demand'):
        print(f"\n===== {self.group_name.upper()} 그룹 스태킹 앙상블 훈련 시작 =====")
        print(f"🎯 {self.season_type} 시즌 최적 파라미터 사용")

        level1_predictions = {}
        oof_predictions = {}

        # 1단계: 개별 모델 훈련 및 Out-of-Fold 예측 생성
        for name, model in self.models.items():
            print(f"--- {name.upper()} 훈련 및 OOF 예측 생성 ---")
            model.best_params = self.predefined_params[name].copy()
            oof_preds_array = np.zeros(len(train_df))

            for fold, (train_idx, val_idx) in enumerate(cv_splits):
                print(f"  Fold {fold+1}/{len(cv_splits)}...")
                fold_train, fold_val = train_df.iloc[train_idx], train_df.iloc[val_idx]

                # 폴드별 모델 훈련
                fold_model_instance = model.__class__(self.season_type)
                fold_model_instance.best_params = model.best_params
                fold_model_instance.fit(fold_train, target_col)

                # 검증 데이터 예측
                fold_preds = fold_model_instance.predict(fold_val)
                oof_preds_array[val_idx] = fold_preds

            oof_predictions[name] = oof_preds_array
            # 전체 데이터로 최종 모델 훈련
            model.fit(train_df, target_col)

        # 2단계: 메타 모델 훈련
        print("--- Ridge 메타 모델 훈련 ---")
        level1_features = np.column_stack(list(oof_predictions.values()))
        targets = train_df[target_col].values

        self.meta_model = Ridge(**self.predefined_params['ridge'], random_state=SEED)
        self.meta_model.fit(level1_features, targets)

        print(f"메타 모델 가중치: Prophet={self.meta_model.coef_[0]:.4f}, CatBoost={self.meta_model.coef_[1]:.4f}")

    def predict(self, test_df):
        print(f"--- {self.group_name.upper()} 그룹 예측 ---")
        level1_predictions = {name: model.predict(test_df) for name, model in self.models.items()}
        meta_features = np.column_stack(list(level1_predictions.values()))
        final_pred = self.meta_model.predict(meta_features)
        return np.maximum(final_pred, 0), level1_predictions

# ---------------------------------------------------
# 5. 메인 실행 로직
# ---------------------------------------------------
if __name__ == "__main__":
    # 1. 데이터 로드 및 전체 전처리
    print("1. 데이터 로드 및 전처리 시작...")
    train_df, test_df = load_and_preprocess_advanced(f"{PATH}train_heat.csv", f"{PATH}test_heat.csv")

    # SVR 보간
    train_df = apply_svr_interpolation(train_df, is_train=True)
    test_df = apply_svr_interpolation(test_df, is_train=False)

    # 이상치 플래그 생성
    train_df, test_df = create_weather_outlier_flags(train_df, test_df)

    # 2. 시즌별 그룹 분할
    train_groups = {'heating': train_df[train_df['heating_season']==1].copy(), 'non_heating': train_df[train_df['heating_season']==0].copy()}
    test_groups = {'heating': test_df[test_df['heating_season']==1].copy(), 'non_heating': test_df[test_df['heating_season']==0].copy()}

    # 3. 그룹별 고급 피처 생성
    print("\n2. 그룹별 피처 엔지니어링 시작...")
    for group_name in ['heating', 'non_heating']:
        print(f"  {group_name} 그룹 피처 생성 중...")
        train_groups[group_name] = create_advanced_features(train_groups[group_name], season_type=group_name)
        test_groups[group_name] = create_advanced_features(test_groups[group_name], season_type=group_name)

    print("전처리 및 피처 엔지니어링 완료!")

    # 4. 그룹별 모델 훈련
    print("\n3. 시즌별 스태킹 모델 훈련 시작...")
    ensemble_models = {}

    def create_year_based_cv_splits(df):
        cv_splits = []
        for val_year in sorted(df['year'].unique())[-3:]:
             train_idx = df[df['year'] != val_year].index
             val_idx = df[df['year'] == val_year].index
             if len(train_idx) > 0 and len(val_idx) > 0:
                 cv_splits.append((train_idx, val_idx))
        if not cv_splits: # 데이터가 부족할 경우 일반 KFold
            kf = KFold(n_splits=3, shuffle=True, random_state=SEED)
            cv_splits = list(kf.split(df))
        return cv_splits

    for group_name, df_group in train_groups.items():
        if len(df_group) > 0:
            cv_splits = create_year_based_cv_splits(df_group)
            ensemble = AdvancedStackingEnsemble(season_type=group_name, group_name=group_name)
            ensemble.fit(df_group, cv_splits)
            ensemble_models[group_name] = ensemble
            joblib.dump(ensemble, f'{PATH}stacking_ensemble_{group_name}.pkl')

    # 5. 테스트 데이터 예측 및 통합
    print("\n4. 테스트 데이터 예측 및 제출 파일 생성...")
    final_predictions = pd.Series(np.zeros(len(test_df)), index=test_df.index)

    for group_name, df_group in test_groups.items():
        if len(df_group) > 0 and group_name in ensemble_models:
            ensemble_model = ensemble_models[group_name]
            preds, _ = ensemble_model.predict(df_group)
            final_predictions.loc[df_group.index] = preds

    # 예측값 후처리
    final_predictions = np.maximum(final_predictions, 0).round(1)

    # 제출 파일 생성
    submission = test_df[['tm', 'branch_id']].copy()
    submission['heat_demand'] = final_predictions.values

    # 최종 제출 파일은 지사별, 시간순으로 정렬
    submission['tm'] = pd.to_datetime(submission['tm'])
    submission_sorted = submission.sort_values(['branch_id', 'tm']).reset_index(drop=True)

    output_filename = "submission_1st_place_target.csv"
    submission_sorted.to_csv(f"{PATH}{output_filename}", index=False)

    print(f"\n🎉 예측 완료! 제출 파일 '{output_filename}'이(가) '{PATH}' 경로에 저장되었습니다.")
    print("제출 파일 샘플:")
    print(submission_sorted.head())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
1. 데이터 로드 및 전처리 시작...
고도화된 데이터 로드 및 전처리...
SVR 보간 적용 중 (TRAIN 데이터)...


지사별 SVR 보간:   0%|          | 0/19 [00:00<?, ?it/s]

SVR 보간 적용 중 (TEST 데이터)...


지사별 SVR 보간:   0%|          | 0/19 [00:00<?, ?it/s]

시즌별 기상 이상치 플래그 생성 중...

2. 그룹별 피처 엔지니어링 시작...
  heating 그룹 피처 생성 중...
  non_heating 그룹 피처 생성 중...
전처리 및 피처 엔지니어링 완료!

3. 시즌별 스태킹 모델 훈련 시작...

===== HEATING 그룹 스태킹 앙상블 훈련 시작 =====
🎯 heating 시즌 최적 파라미터 사용
--- PROPHET 훈련 및 OOF 예측 생성 ---
  Fold 1/3...


IndexError: positional indexers are out-of-bounds

# 마지막 실험

코드 주요 개선사항 요약
create_advanced_features 함수 대폭 강화:

논문의 핵심 아이디어인 이력 현상, 심리적 요인, 동적 특성을 반영한 파생변수들을 체계적으로 추가했습니다.

풍향(wd)과 같은 순환 변수를 sin/cos 변환하여 모델이 패턴을 더 잘 학습하도록 했습니다.

타겟 변수 로그 변환:

열 수요와 같이 오른쪽으로 꼬리가 긴 분포를 가진 데이터는 로그 변환(np.log1p) 시 모델 성능이 안정화되고 RMSE가 개선되는 경우가 많습니다. 예측 후에는 다시 원래 스케일(np.expm1)로 복원합니다.

단조성 제약(Monotonic Constraints) 적용:

get_monotone_constraints 함수를 추가하고 model.fit 시점에 적용하여, 모델이 상식에 어긋나지 않는 예측을 하도록 유도했습니다.

과거 수요(Lag Feature) 생성 로직 개선:

테스트 데이터에 대해서도 과거 수요 정보를 활용할 수 있도록, 훈련 데이터의 마지막 부분을 테스트 데이터와 결합하여 시차 변수를 생성하는 로직을 추가했습니다.

In [2]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import joblib
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
import optuna
from tqdm.auto import tqdm
import warnings
import holidays
import os

warnings.filterwarnings('ignore')

# Google Colab 사용 시 경로 설정 (로컬 환경에서는 PATH = "./"로 설정)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    PATH = "/content/drive/MyDrive/열수요예측/"
    if not os.path.exists(PATH):
        os.makedirs(PATH)
except (ImportError, ModuleNotFoundError):
    PATH = "./"

# 시드 고정
SEED = 42
np.random.seed(SEED)

def load_and_preprocess(df):
    """기본 컬럼명 정리 및 타입 변환"""
    df = df.copy()
    if 'Unnamed: 0' in df.columns:
        df = df.drop(columns=['Unnamed: 0'])
    df.columns = [col.lower().replace('train_heat.', '').replace('test_heat.', '') for col in df.columns]

    if 'tm' not in df.columns:
        time_col = [col for col in df.columns if 'tm' in col]
        if time_col:
            df.rename(columns={time_col[0]: 'tm'}, inplace=True)
        else:
            raise KeyError("데이터에서 'tm' 컬럼을 찾을 수 없습니다.")

    df['tm'] = pd.to_datetime(df['tm'], errors='coerce')
    if df['tm'].isnull().any():
        df['tm'] = pd.to_datetime(df['tm'], format='%Y%m%d%H', errors='coerce')

    if 'branch_id' in df.columns:
        df['branch_id'] = df['branch_id'].astype('category')
    return df

def create_missing_flags_and_replace(df):
    """결측치 플래그 생성 및 NaN으로 변환"""
    df = df.copy()
    missing_cols = ['ta', 'ws', 'rn_day', 'rn_hr1', 'hm', 'si', 'ta_chi', 'heat_demand']
    for col in missing_cols:
        if col in df.columns:
            df[f'{col}_missing'] = (df[col] == -99).astype(int)
            df[col] = df[col].replace(-99, np.nan)
    if 'wd' in df.columns:
        df['wd'] = df['wd'].replace(-9.9, np.nan)
    return df

def apply_interpolation(df):
    """지사별 시계열 보간"""
    df = df.copy()
    interp_cols = ['ta', 'ws', 'rn_day', 'rn_hr1', 'hm', 'si', 'ta_chi', 'wd']
    if 'heat_demand' in df.columns:
        interp_cols.append('heat_demand')

    current_interp_cols = [col for col in interp_cols if col in df.columns]
    df = df.sort_values(by=['branch_id', 'tm']).reset_index(drop=True)
    df[current_interp_cols] = df.groupby('branch_id')[current_interp_cols].transform(
        lambda group: group.interpolate(method='linear', limit_direction='both')
    )
    # 보간 후 남은 결측치는 0으로 채움
    df[current_interp_cols] = df[current_interp_cols].fillna(0)
    return df

def create_log_target(df):
    """타겟 변수 로그 변환 (분포 안정화)"""
    df = df.copy()
    if 'heat_demand' in df.columns:
        # 0 이하의 값을 처리하기 위해 np.log1p 사용
        df['heat_demand'] = np.log1p(df['heat_demand'])
    return df

def create_advanced_features(df):
    """고도화된 파생변수 생성 (논문 아이디어 반영)"""
    df = df.copy()
    df['month'] = df['tm'].dt.month
    df['day'] = df['tm'].dt.day
    df['hour'] = df['tm'].dt.hour
    df['dayofweek'] = df['tm'].dt.dayofweek
    df['dayofyear'] = df['tm'].dt.dayofyear
    df['quarter'] = df['tm'].dt.quarter

    # 순환 인코딩
    df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
    df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)
    df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
    df['dayofweek_sin'] = np.sin(2 * np.pi * df['dayofweek'] / 7)

    # ==========================================================
    # [개선] 논문 인사이트 반영 피처
    # ==========================================================
    # 1. 이력현상(Hysteresis) 및 상태 의존성 피처 (서병선 외, 2012)
    temp_threshold = 18.0  # 난방도일 기준 및 논문에서 제시된 임계점 근사
    df['is_above_threshold'] = (df['ta'] > temp_threshold).astype(int)
    df['temp_x_interaction'] = df['ta'] * df['is_above_threshold']

    # 2. '심리적 요인' 반영 기온 변화량 피처 (서한석 외, 2018)
    df['ta_diff_24h'] = df.groupby('branch_id')['ta'].diff(24)

    # 3. 고급 기상 변수 및 상호작용
    df['HDD18'] = np.maximum(0, 18 - df['ta'])  # 난방도일
    df['apparent_temp'] = 13.12 + 0.6215 * df['ta'] - 11.37 * (df['ws'] * 3.6)**0.16 + 0.3965 * df['ta'] * (df['ws'] * 3.6)**0.16
    df['discomfort_index'] = 1.8 * df['ta'] - 0.55 * (1 - df['hm']/100) * (1.8 * df['ta'] - 26) + 32 # 불쾌지수
    df['temp_x_humidity'] = df['ta'] * df['hm']

    # 4. 풍향(wd) 변수 -> sin/cos 변환
    df['wd_sin'] = np.sin(2 * np.pi * df['wd'] / 360)
    df['wd_cos'] = np.cos(2 * np.pi * df['wd'] / 360)

    # 시차 및 이동평균 (지사별)
    lags = [24, 48, 72] # 1일, 2일, 3일 전
    windows = [6, 12, 24]

    # [개선] 과거 수요(heat_demand) 시차 변수 추가
    if 'heat_demand' in df.columns:
        for lag in lags:
            df[f'heat_demand_lag_{lag}h'] = df.groupby('branch_id')['heat_demand'].shift(lag)

    for lag in lags:
        df[f'ta_lag_{lag}h'] = df.groupby('branch_id')['ta'].shift(lag)
    for window in windows:
        df[f'ta_ma_{window}h'] = df.groupby('branch_id')['ta'].transform(lambda x: x.rolling(window, min_periods=1).mean())
        df[f'ta_std_{window}h'] = df.groupby('branch_id')['ta'].transform(lambda x: x.rolling(window, min_periods=1).std())


    # 공휴일
    kr_holidays = holidays.KR()
    df['is_holiday'] = df['tm'].dt.date.apply(lambda x: x in kr_holidays).astype(int)

    # 파생변수 생성으로 인한 결측치 다시 채우기
    feature_cols_for_fillna = [col for col in df.columns if col not in ['tm', 'heat_demand', 'branch_id']]
    df[feature_cols_for_fillna] = df.groupby('branch_id')[feature_cols_for_fillna].transform(lambda x: x.ffill().bfill())
    df[feature_cols_for_fillna] = df[feature_cols_for_fillna].fillna(0)
    return df

def get_feature_names(df):
    """사용할 피처 이름 정의"""
    features = [col for col in df.columns if col not in ['tm', 'heat_demand', 'year']]
    categorical_features = [
        'branch_id', 'month', 'hour', 'dayofweek', 'heating_season',
        'is_holiday', 'quarter', 'is_above_threshold'
    ]
    categorical_features = [f for f in categorical_features if f in df.columns]
    return features, categorical_features

def get_monotone_constraints(feature_names):
    """LightGBM 단조성 제약 설정 (도메인 지식 주입)"""
    constraints_dict = {
        'ta': -1, 'apparent_temp': -1, 'discomfort_index': -1, 'si': -1,
        'HDD18': 1, 'ws': 1
    }
    constraints = [0] * len(feature_names)
    for i, feature in enumerate(feature_names):
        for key, value in constraints_dict.items():
            if key in feature and 'lag' not in feature and 'std' not in feature:
                constraints[i] = value
                break
    return constraints

def optimize_and_train(df_group, group_name):
    """Optuna로 하이퍼파라미터 최적화 및 모델 훈련"""
    print(f"\n===== {group_name.upper()} 그룹 모델 훈련 시작 =====")

    X = df_group.drop(columns=['tm', 'heat_demand'])
    y = df_group['heat_demand']
    features, categorical_features = get_feature_names(X)
    X = X[features]

    # 연도 기반 CV 분할
    kf = KFold(n_splits=3, shuffle=True, random_state=SEED)
    cv_splits = list(kf.split(X))

    def objective(trial):
        params = {
            'objective': 'regression_l1', 'metric': 'rmse', 'n_estimators': 10000,
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.05),
            'num_leaves': trial.suggest_int('num_leaves', 31, 256),
            'max_depth': trial.suggest_int('max_depth', 7, 15),
            'min_child_samples': trial.suggest_int('min_child_samples', 20, 100),
            'subsample': trial.suggest_float('subsample', 0.7, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.7, 1.0),
            'random_state': SEED, 'n_jobs': -1, 'verbose': -1,
        }

        cv_rmses = []
        for train_idx, val_idx in cv_splits:
            X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
            y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

            model = lgb.LGBMRegressor(**params)
            model.fit(X_train, y_train,
                      eval_set=[(X_val, y_val)],
                      eval_metric='rmse',
                      callbacks=[lgb.early_stopping(100, verbose=False)],
                      categorical_feature=categorical_features)

            preds = model.predict(X_val)
            # 로그 변환된 타겟에 대한 RMSE 계산
            cv_rmses.append(np.sqrt(mean_squared_error(y_val, preds)))
        return np.mean(cv_rmses)

    study = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=SEED))
    study.optimize(objective, n_trials=30, show_progress_bar=True)
    print(f"   {group_name} 그룹 최적 RMSE (log scale): {study.best_value:.4f}")

    best_params = study.best_params
    best_params.update({'objective': 'regression_l1', 'metric': 'rmse', 'random_state': SEED, 'n_estimators': 10000})

    final_model = lgb.LGBMRegressor(**best_params)

    # [개선] 단조성 제약 추가
    monotone_constraints = get_monotone_constraints(features)

    final_model.fit(X, y,
                    categorical_feature=categorical_features,
                    eval_set=[(X, y)],
                    callbacks=[lgb.early_stopping(100), lgb.log_evaluation(1000)],
                    monotone_constraints=monotone_constraints)

    joblib.dump(final_model, f'{PATH}lgb_model_{group_name}.pkl')
    joblib.dump(features, f'{PATH}features_{group_name}.pkl')
    return final_model

if __name__ == "__main__":
    print("1. 데이터 로드 및 전처리 시작...")
    train_df_raw = pd.read_csv(PATH + "train_heat.csv")
    test_df_raw = pd.read_csv(PATH + "test_heat.csv")

    train_df = load_and_preprocess(train_df_raw)
    test_df = load_and_preprocess(test_df_raw)

    train_df = create_missing_flags_and_replace(train_df)
    test_df = create_missing_flags_and_replace(test_df)

    train_df = apply_interpolation(train_df)
    test_df = apply_interpolation(test_df)

    # [개선] 타겟 로그 변환 적용
    train_df = create_log_target(train_df)

    # [개선] 시즌 컬럼을 먼저 생성
    train_df['heating_season'] = train_df['tm'].dt.month.isin([10, 11, 12, 1, 2, 3, 4]).astype(int)
    test_df['heating_season'] = test_df['tm'].dt.month.isin([10, 11, 12, 1, 2, 3, 4]).astype(int)

    # [개선] 모든 고급 피처 생성
    train_df = create_advanced_features(train_df)
    # 테스트 데이터 예측 시 lag_heat_demand 생성을 위해 train_df의 마지막 부분을 활용
    combined_df = pd.concat([train_df.tail(72), test_df], ignore_index=True)
    combined_df_processed = create_advanced_features(combined_df)
    test_df = combined_df_processed.iloc[72:].reset_index(drop=True)

    print("전처리 완료!")

    print("\n2. 시즌별 모델 훈련 시작...")
    train_groups = {name: data for name, data in train_df.groupby('heating_season')}
    models = {}
    for group_id, df_group in train_groups.items():
        group_name = 'heating' if group_id == 1 else 'non_heating'
        if len(df_group) > 0:
            models[group_name] = optimize_and_train(df_group, group_name)

    print("\n3. 테스트 데이터 예측 및 제출 파일 생성...")
    test_groups = {name: data for name, data in test_df.groupby('heating_season')}
    final_predictions = pd.Series(np.nan, index=test_df.index)

    for group_id, df_group in test_groups.items():
        group_name = 'heating' if group_id == 1 else 'non_heating'
        if len(df_group) > 0 and group_name in models:
            print(f"   {group_name} 그룹 예측 중...")
            model = models[group_name]
            train_features = joblib.load(f'{PATH}features_{group_name}.pkl')
            X_test_group = df_group.reindex(columns=train_features, fill_value=0)

            preds_log = model.predict(X_test_group)

            # [개선] 로그 변환된 예측값을 원래 스케일로 복원
            preds = np.expm1(preds_log)

            final_predictions.loc[df_group.index] = preds

    final_predictions = np.maximum(final_predictions, 0) # 0보다 작은 값은 0으로

    submission = test_df_raw[['TM', 'branch_ID']].copy()
    submission['heat_demand'] = final_predictions

    output_filename = "submission_lgbm_expert_tuned.csv"
    submission.to_csv(f"{PATH}{output_filename}", index=False)

    print(f"\n🎉 예측 완료! 제출 파일 '{output_filename}'이(가) '{PATH}' 경로에 저장되었습니다.")

Mounted at /content/drive
1. 데이터 로드 및 전처리 시작...
전처리 완료!

2. 시즌별 모델 훈련 시작...

===== HEATING 그룹 모델 훈련 시작 =====


[I 2025-06-25 16:21:08,246] A new study created in memory with name: no-name-ef62b68e-fe3a-42f5-a958-278e717757dd


  0%|          | 0/30 [00:00<?, ?it/s]

[W 2025-06-25 16:22:57,244] Trial 0 failed with parameters: {'learning_rate': 0.0249816047538945, 'num_leaves': 245, 'max_depth': 13, 'min_child_samples': 68, 'subsample': 0.7468055921327309, 'colsample_bytree': 0.7467983561008608} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "/usr/local/lib/python3.11/dist-packages/optuna/study/_optimize.py", line 201, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "/tmp/ipython-input-2-1254600611.py", line 205, in objective
    model.fit(X_train, y_train,
  File "/usr/local/lib/python3.11/dist-packages/lightgbm/sklearn.py", line 1189, in fit
    super().fit(
  File "/usr/local/lib/python3.11/dist-packages/lightgbm/sklearn.py", line 955, in fit
    self._Booster = train(
                    ^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/lightgbm/engine.py", line 307, in train
    booster.update(fobj=fobj)
  File "/usr/local/lib/python3.11/dist-packages/

KeyboardInterrupt: 